# Feature Engineering — ChemAI

Демонстрация и обоснование всех трансформаций признаков,
применяемых в `solution.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

SEED            = 42
CORR_THRESHOLD  = 0.95
TARGET_IC50     = 'IC50, mM'
TARGET_CC50     = 'CC50, mM'
TARGET_SI       = 'SI'
TARGETS         = [TARGET_IC50, TARGET_CC50, TARGET_SI]
POSITIVE_FEATS  = [
    'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'TPSA',
    'LabuteASA', 'BertzCT', 'Ipc', 'Chi0', 'Chi0n', 'Chi0v', 'MolMR',
]

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
print(f'Train: {train.shape}, Test: {test.shape}')

## 1. Удаление константных признаков

In [ ]:
feat_cols   = [c for c in train.columns if c not in ['index'] + TARGETS]
const_feats = [c for c in feat_cols if train[c].nunique() <= 1]
feat_cols   = [c for c in feat_cols if c not in const_feats]

print(f'Удалено константных: {len(const_feats)}')
print('Константные признаки:', const_feats)

## 2. Заполнение пропусков

Fit только на train — исключает утечку из test.

In [ ]:
missing = train[feat_cols].isnull().sum()
print('Признаки с пропусками до заполнения:')
print(missing[missing > 0])

imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(train[feat_cols]), columns=feat_cols)
X_test  = pd.DataFrame(imputer.transform(test[feat_cols]),      columns=feat_cols)
print(f'\nПропусков после заполнения: {X_train.isnull().sum().sum()}')

## 3. Log-дополнение положительных дескрипторов

Молекулярные дескрипторы типа MolWt логнормально распределены.
Добавление log-версий помогает бустинговым моделям находить нелинейности.

In [ ]:
added = []
for feat in POSITIVE_FEATS:
    if feat in feat_cols and (X_train[feat] > 0).all():
        X_train[f'log_{feat}'] = np.log(X_train[feat])
        X_test[f'log_{feat}']  = np.log(X_test[feat])
        added.append(feat)

print(f'Добавлено log-признаков: {len(added)}')
print(f'Итого признаков: {X_train.shape[1]}')

# Наглядная демонстрация: MolWt vs log(MolWt)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(X_train['MolWt'], bins=40, edgecolor='k', alpha=0.7)
ax1.set_title('MolWt (original)')
ax2.hist(X_train['log_MolWt'], bins=40, edgecolor='k', alpha=0.7, color='orange')
ax2.set_title('log(MolWt)')
plt.tight_layout()
plt.show()

## 4. Удаление коррелированных признаков (r > 0.95)

Высококоррелированные признаки несут одинаковую информацию и добавляют шум.
Порог 0.95 удаляет явных дубликатов, сохраняя разнообразие.

In [ ]:
corr_m    = X_train.corr().abs()
upper     = corr_m.where(np.triu(np.ones(corr_m.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]

X_train = X_train.drop(columns=drop_corr)
X_test  = X_test.drop(columns=drop_corr)

print(f'Удалено коррелированных: {len(drop_corr)}')
print(f'Итоговых признаков: {X_train.shape[1]}')
print('Примеры удалённых:', drop_corr[:10])

## 5. Итоговый пайплайн препроцессинга

In [ ]:
print('Пайплайн признаков:')
print(f'  Исходно:                    {len([c for c in train.columns if c not in ["index"] + TARGETS])}')
print(f'  После удаления константных: {len(feat_cols)}')
print(f'  После log-дополнения:       {len(feat_cols) + len(added)}')
print(f'  После фильтра корреляций:   {X_train.shape[1]}')
print()
print('Утечка из test: НЕТ (imputer.fit только на train)')
print('SEED для воспроизводимости: используется в моделях, не в препроцессинге')